# 03 — Volatility targeting & position sizing

**Project:** size positions so realised risk tracks a target portfolio volatility.

## Problem
A raw ±1 signal treats a 10% vol asset the same as a 40% vol asset. Risk — and
drawdowns — then explode when volatility rises. How do we normalise exposure?

## Method
1. Estimate rolling asset volatility from synthetic returns.
2. Size with `vol_target(capital, target_vol, asset_vol, price)`.
3. Compare unscaled vs vol-targeted equity and drawdowns via `risk_summary`.

## Modules
- `quant_utils.risk.sizing`
- `quant_utils.risk.metrics`
- `quant_utils.data.sample`
- `quant_utils.backtest` (for a simple long-only demo)


In [ ]:
import numpy as np
import pandas as pd
from quant_utils.data import make_ohlcv
from quant_utils.risk import vol_target, risk_summary, sharpe, max_drawdown
from quant_utils.backtest import run_backtest, summary_table

df = make_ohlcv(n=504, drift=0.0002, vol=0.015, seed=11)
rets = df["Close"].pct_change()

# Rolling annualised vol estimate (21-day)
roll_vol = rets.rolling(21).std() * np.sqrt(252)

capital = 100_000.0
target_vol = 0.10  # 10% annualised portfolio vol target
prices = df["Close"]

shares = []
for px, av in zip(prices, roll_vol):
    if not np.isfinite(av) or av <= 0:
        shares.append(0)
    else:
        shares.append(vol_target(capital, target_vol, float(av), float(px)))

size = pd.Series(shares, index=df.index, name="shares")
# Convert share count to a fractional position vs fully invested capital
notional = size * prices
frac = (notional / capital).clip(upper=1.5).fillna(0.0)
print(size.tail())
print("median fraction of capital:", float(frac.replace(0, np.nan).median()))


In [ ]:
# Always-long benchmark vs vol-targeted long
sig_full = pd.Series(1.0, index=df.index)
sig_vt = frac  # fractional long sized to target vol

res_full = run_backtest(sig_full, prices, slippage=0.0002, commission=0.0005)
res_vt = run_backtest(sig_vt, prices, slippage=0.0002, commission=0.0005)

print("Fully invested long")
print(summary_table(res_full))
print()
print("Vol-targeted long")
print(summary_table(res_vt))
print()
print("Realised ann. vol (full):", float(res_full.returns.std() * np.sqrt(252)))
print("Realised ann. vol (VT):  ", float(res_vt.returns.std() * np.sqrt(252)))


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-friendly
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
res_full.equity_curve.plot(ax=axes[0], label="Full long")
res_vt.equity_curve.plot(ax=axes[0], label="Vol target")
axes[0].set_ylabel("Equity")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
frac.plot(ax=axes[1], color="tab:orange")
axes[1].set_ylabel("Position fraction")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Short result
Vol targeting pulls exposure down when rolling vol is high, so realised strategy
volatility sits closer to the 10% target than a fully invested long (see printed
ann. vols). Drawdowns are typically milder at the cost of lower upside in calm trends.

## Limitations
- Rolling vol is a lagging estimator; jumps still hit before size adjusts.
- `vol_target` floors to integer shares — coarse for expensive underlyings.
- Ignores correlation in multi-asset books; single-name demo only.
- Research/education only.
